In [45]:
import pandas as pd
import numpy as np
sales= pd.read_csv("Final_Feature_code.csv")
#sales.dropna(inplace= True)
#sales.isnull().sum()

In [46]:
sales.shape

(100, 1033)

In [47]:
sales.head(100)

,Unnamed: 0,customer_id,order_date,purchase_count,avg_quantity,max_quantity,p25_quantity,p50_quantity,p75_quantity,p90_quantity,...,max_pp_30d_mean,max_pp_30d_min,max_pp_30d_max,max_pp_30d_std,max_pp_30d_var,max_pp_30d_p25,max_pp_30d_p50,max_pp_30d_p75,max_pp_30d_p90,max_pp_30d_p95
0,0,804,2026-04-21,1,1.0,1,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1079,2026-01-12,1,1.0,1,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,1325,2026-01-18,1,1.0,1,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,1987,2026-02-01,1,1.0,1,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,2593,2026-01-09,1,1.0,1,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,95210,2026-04-03,1,25.0,25,25.0,25.0,25.0,25.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
96,96,96627,2026-03-05,1,25.0,25,25.0,25.0,25.0,25.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
97,97,96764,2026-01-18,1,25.0,25,25.0,25.0,25.0,25.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
98,98,96876,2026-01-23,1,25.0,25,25.0,25.0,25.0,25.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [48]:
sales.fillna(sales.median(numeric_only=True), inplace=True)
sales.dropna(inplace= True)
sales.shape

(0, 1033)

In [49]:
#sales.drop(*['order_date', 'last_order_date'],axis=1, inplace= True)
sales.drop('order_date',axis=1, inplace= True)
sales.drop('customer_id',axis=1, inplace= True)
sales.drop('Unnamed: 0',axis=1, inplace= True)

In [50]:
sales.columns

Index(['purchase_count', 'avg_quantity', 'max_quantity', 'p25_quantity',
       'p50_quantity', 'p75_quantity', 'p90_quantity', 'p95_quantity',
       'total_spend', 'avg_spend',
       ...
       'max_pp_30d_mean', 'max_pp_30d_min', 'max_pp_30d_max', 'max_pp_30d_std',
       'max_pp_30d_var', 'max_pp_30d_p25', 'max_pp_30d_p50', 'max_pp_30d_p75',
       'max_pp_30d_p90', 'max_pp_30d_p95'],
      dtype='object', length=1030)

In [51]:
sales.shape

(0, 1030)

In [55]:
# ============================================================
# COMPLETE MULTICLASS LIGHTGBM + SHAP ANALYSIS
# FOR ~500 PRODUCT CLASSES
# ============================================================
#
# Expected:
#   ~1,030 rows
#   ~1,000+ engineered features
#   ~500 product classes
#
# Target:
#   product_id
#
# IMPORTANT:
#   sales["product_id"] is currently EMPTY in your output.
#   This code searches for a dataframe containing a valid
#   product_id before proceeding.
#
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================

import os
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import shap

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    top_k_accuracy_score
)

from lightgbm import LGBMClassifier


# ============================================================
# 1. SETTINGS
# ============================================================

TARGET = "product_id"

TEST_SIZE = 0.20

RANDOM_STATE = 42

OUTPUT_DIR = "multiclass_product_shap_results"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ------------------------------------------------------------
# SHAP SETTINGS
# ------------------------------------------------------------

# Number of rows used for SHAP.
#
# You only have ~1,030 rows, so this is reasonable.
#
SHAP_SAMPLE_SIZE = 300

# Number of products to investigate individually.
TOP_PRODUCTS_TO_ANALYZE = 10

# Number of features shown per product.
TOP_FEATURES_PER_PRODUCT = 20

# Number of global features.
TOP_GLOBAL_FEATURES = 30


# ============================================================
# 2. HEADER
# ============================================================

print("=" * 90)
print("MULTICLASS LIGHTGBM + SHAP")
print("PRODUCT PREDICTION")
print("=" * 90)


# ============================================================
# 3. FIND DATAFRAME WITH VALID PRODUCT_ID
# ============================================================

print("\n" + "=" * 90)
print("SEARCHING FOR VALID DATAFRAME")
print("=" * 90)


candidate_names = [
    "sales",
    "customer_product_date",
    "cpd",
    "df"
]


selected_df = None
selected_name = None


for name in candidate_names:

    if name not in globals():
        continue

    temp = globals()[name]

    if not isinstance(temp, pd.DataFrame):
        continue

    print(
        f"\n{name}: shape = {temp.shape}"
    )

    if TARGET not in temp.columns:

        print(
            f"   {TARGET}: COLUMN NOT FOUND"
        )

        continue

    non_null = temp[TARGET].notna().sum()
    unique = temp[TARGET].nunique()

    print(
        f"   {TARGET} non-null = {non_null}"
    )

    print(
        f"   {TARGET} unique = {unique}"
    )

    if non_null > 0 and unique >= 2:

        selected_df = temp.copy()

        selected_name = name

        print(
            f"\n>>> SELECTED DATAFRAME: {name}"
        )

        break


# ============================================================
# 4. STOP IF NO VALID DATAFRAME
# ============================================================

if selected_df is None:

    raise ValueError(
        """
NO DATAFRAME WITH A VALID product_id WAS FOUND.

Your current sales dataframe appears to have:

    product_id non-null = 0

Therefore the model cannot train.

You need to restore product_id from the dataframe used
to create your customer-product feature store.

Check:

    sales["product_id"].value_counts(dropna=False)

    customer_product_date["product_id"].value_counts(dropna=False)

    cpd["product_id"].value_counts(dropna=False)

"""
    )


df = selected_df.copy()


print(
    "\nUsing dataframe:",
    selected_name
)

print(
    "Shape:",
    df.shape
)


# ============================================================
# 5. BASIC VALIDATION
# ============================================================

print("\n" + "=" * 90)
print("DATA VALIDATION")
print("=" * 90)


print(
    "\nOriginal shape:",
    df.shape
)

print(
    "Product ID dtype:",
    df[TARGET].dtype
)

print(
    "Product ID non-null:",
    df[TARGET].notna().sum()
)

print(
    "Product ID null:",
    df[TARGET].isna().sum()
)

print(
    "Number of products:",
    df[TARGET].nunique()
)


if df[TARGET].nunique() < 2:

    raise ValueError(
        "product_id must contain at least 2 classes."
    )


# ============================================================
# 6. REMOVE NULL TARGET
# ============================================================

df = df.dropna(
    subset=[TARGET]
).copy()


print(
    "\nAfter removing NULL product_id:"
)

print(
    "Rows:",
    len(df)
)

print(
    "Products:",
    df[TARGET].nunique()
)


# ============================================================
# 7. CLASS DISTRIBUTION
# ============================================================

class_counts = (
    df[TARGET]
    .value_counts()
    .sort_values()
)


print("\n" + "=" * 90)
print("PRODUCT CLASS DISTRIBUTION")
print("=" * 90)


print(
    "\nNumber of product classes:",
    len(class_counts)
)

print(
    "\nSmallest classes:"
)

print(
    class_counts.head(20)
)

print(
    "\nLargest classes:"
)

print(
    class_counts.tail(20)
)


print(
    "\nClasses with < 2 observations:",
    (class_counts < 2).sum()
)

print(
    "Classes with < 5 observations:",
    (class_counts < 5).sum()
)

print(
    "Classes with < 10 observations:",
    (class_counts < 10).sum()
)


# ============================================================
# 8. REMOVE SINGLETON CLASSES
# ============================================================
#
# A stratified train/test split cannot reliably work with
# classes having only one observation.
#
# ============================================================

valid_classes = class_counts[
    class_counts >= 2
].index


removed_classes = class_counts[
    class_counts < 2
]


df = df[
    df[TARGET].isin(valid_classes)
].copy()


print("\n" + "=" * 90)
print("AFTER REMOVING SINGLETON PRODUCTS")
print("=" * 90)


print(
    "Rows:",
    len(df)
)

print(
    "Products:",
    df[TARGET].nunique()
)

print(
    "Removed classes:",
    len(removed_classes)
)


if len(df) == 0:

    raise ValueError(
        "No rows remain after removing singleton classes."
    )


if df[TARGET].nunique() < 2:

    raise ValueError(
        "Fewer than 2 product classes remain."
    )


# ============================================================
# 9. SEPARATE X AND Y
# ============================================================

X = df.drop(
    columns=[TARGET]
).copy()

y = df[TARGET].copy()


print("\n" + "=" * 90)
print("FEATURE DATA")
print("=" * 90)


print(
    "Rows:",
    X.shape[0]
)

print(
    "Features:",
    X.shape[1]
)


# ============================================================
# 10. REMOVE CONSTANT FEATURES
# ============================================================
#
# With ~1,000 engineered features and only ~1,030 rows,
# constant columns are useless.
#
# ============================================================

constant_columns = [
    col
    for col in X.columns
    if X[col].nunique(dropna=False) <= 1
]


print(
    "\nConstant features:",
    len(constant_columns)
)


if constant_columns:

    X = X.drop(
        columns=constant_columns
    )


print(
    "Features after constant removal:",
    X.shape[1]
)


# ============================================================
# 11. REMOVE DUPLICATE FEATURES
# ============================================================
#
# Optional but useful with many engineered rolling features.
#
# ============================================================

duplicate_columns = []

seen = {}


for col in X.columns:

    try:

        key = tuple(
            X[col].fillna("__NA__").astype(str)
        )

        if key in seen:

            duplicate_columns.append(col)

        else:

            seen[key] = col

    except Exception:

        pass


print(
    "\nDuplicate feature columns:",
    len(duplicate_columns)
)


if duplicate_columns:

    X = X.drop(
        columns=duplicate_columns
    )


print(
    "Features after duplicate removal:",
    X.shape[1]
)


# ============================================================
# 12. HANDLE DATETIME FEATURES
# ============================================================

datetime_cols = X.select_dtypes(
    include=[
        "datetime64[ns]",
        "datetime64[ns, UTC]"
    ]
).columns.tolist()


print("\nDatetime columns:")

print(
    datetime_cols
)


for col in datetime_cols:

    X[col + "_year"] = X[col].dt.year

    X[col + "_month"] = X[col].dt.month

    X[col + "_day"] = X[col].dt.day

    X[col + "_dayofweek"] = X[col].dt.dayofweek

    X[col + "_dayofyear"] = X[col].dt.dayofyear

    X[col + "_week"] = (
        X[col]
        .dt.isocalendar()
        .week
        .astype(float)
    )

    X[col + "_is_weekend"] = (
        X[col].dt.dayofweek >= 5
    ).astype(int)

    X.drop(
        columns=[col],
        inplace=True
    )


# ============================================================
# 13. HANDLE OBJECT / CATEGORY COLUMNS
# ============================================================

categorical_cols = X.select_dtypes(
    include=[
        "object",
        "category"
    ]
).columns.tolist()


print(
    "\nCategorical features:",
    len(categorical_cols)
)


for col in categorical_cols:

    X[col] = X[col].astype("category")


# ============================================================
# 14. REPLACE INF
# ============================================================

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)


# ============================================================
# 15. FINAL FEATURE INFORMATION
# ============================================================

numeric_cols = X.select_dtypes(
    include=np.number
).columns.tolist()


categorical_cols = X.select_dtypes(
    include="category"
).columns.tolist()


print("\n" + "=" * 90)
print("FINAL FEATURES")
print("=" * 90)


print(
    "Numeric:",
    len(numeric_cols)
)

print(
    "Categorical:",
    len(categorical_cols)
)

print(
    "Total:",
    X.shape[1]
)


# ============================================================
# 16. ENCODE TARGET
# ============================================================
#
# Using integer labels is safer for multiclass LightGBM.
#
# ============================================================

label_encoder = LabelEncoder()


y_encoded = label_encoder.fit_transform(
    y
)


classes = label_encoder.classes_


NUM_CLASSES = len(classes)


print("\n" + "=" * 90)
print("TARGET ENCODING")
print("=" * 90)


print(
    "Number of classes:",
    NUM_CLASSES
)


# ============================================================
# 17. TRAIN TEST SPLIT
# ============================================================

print("\n" + "=" * 90)
print("TRAIN / TEST SPLIT")
print("=" * 90)


X_train, X_test, y_train, y_test = train_test_split(

    X,

    y_encoded,

    test_size=TEST_SIZE,

    random_state=RANDOM_STATE,

    stratify=y_encoded
)


print(
    "\nTraining rows:",
    len(X_train)
)

print(
    "Testing rows:",
    len(X_test)
)

print(
    "Training features:",
    X_train.shape[1]
)

print(
    "Training classes:",
    len(np.unique(y_train))
)

print(
    "Testing classes:",
    len(np.unique(y_test))
)


# ============================================================
# 18. CHECK CLASSES
# ============================================================

train_classes = set(
    np.unique(y_train)
)

test_classes = set(
    np.unique(y_test)
)


missing_classes = (
    test_classes - train_classes
)


if missing_classes:

    print(
        "\nWARNING: classes missing from training:"
    )

    print(
        [
            classes[i]
            for i in missing_classes
        ]
    )

else:

    print(
        "\nAll test classes exist in training."
    )


# ============================================================
# 19. LIGHTGBM MODEL
# ============================================================

print("\n" + "=" * 90)
print("LIGHTGBM")
print("=" * 90)


model = LGBMClassifier(

    objective="multiclass",

    num_class=NUM_CLASSES,

    n_estimators=300,

    learning_rate=0.03,

    num_leaves=15,

    max_depth=8,

    min_child_samples=10,

    subsample=0.8,

    colsample_bytree=0.7,

    reg_alpha=2.0,

    reg_lambda=5.0,

    random_state=RANDOM_STATE,

    n_jobs=-1,

    verbosity=-1
)


# ============================================================
# 20. TRAIN
# ============================================================

print(
    "\nTraining LightGBM..."
)


model.fit(

    X_train,

    y_train,

    categorical_feature=categorical_cols
)


print(
    "Training completed."
)


# ============================================================
# 21. PREDICTION
# ============================================================

print(
    "\nGenerating predictions..."
)


y_pred_encoded = model.predict(
    X_test
)


y_prob = model.predict_proba(
    X_test
)


y_pred = label_encoder.inverse_transform(
    y_pred_encoded.astype(int)
)


y_test_original = label_encoder.inverse_transform(
    y_test.astype(int)
)


print(
    "Probability matrix:",
    y_prob.shape
)


# ============================================================
# 22. PERFORMANCE
# ============================================================

print("\n" + "=" * 90)
print("MODEL PERFORMANCE")
print("=" * 90)


accuracy = accuracy_score(
    y_test,
    y_pred_encoded
)


balanced_accuracy = balanced_accuracy_score(
    y_test,
    y_pred_encoded
)


print(
    "\nAccuracy:",
    round(accuracy, 4)
)


print(
    "Balanced Accuracy:",
    round(balanced_accuracy, 4)
)


# ============================================================
# 23. TOP-K ACCURACY
# ============================================================

print("\nTop-K Accuracy:")


for k in [3, 5, 10]:

    try:

        score = top_k_accuracy_score(

            y_test,

            y_prob,

            k=k,

            labels=np.arange(NUM_CLASSES)

        )

        print(
            f"Top-{k}:",
            round(score, 4)
        )

    except Exception as e:

        print(
            f"Top-{k}: unavailable"
        )


# ============================================================
# 24. CLASSIFICATION REPORT
# ============================================================

report = classification_report(

    y_test,

    y_pred_encoded,

    labels=np.arange(NUM_CLASSES),

    target_names=[
        str(x)
        for x in classes
    ],

    digits=4,

    zero_division=0
)


print("\nClassification Report:\n")

print(report)


with open(

    os.path.join(
        OUTPUT_DIR,
        "classification_report.txt"
    ),

    "w"

) as f:

    f.write(report)


# ============================================================
# 25. PREDICTION RESULTS
# ============================================================

prediction_results = X_test.copy()


prediction_results["actual_product_id"] = (
    y_test_original
)


prediction_results["predicted_product_id"] = (
    y_pred
)


prediction_results["prediction_confidence"] = (
    y_prob.max(axis=1)
)


prediction_results["correct_prediction"] = (

    y_test_original == y_pred

)


prediction_results.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "test_predictions.csv"
    ),

    index=True
)


# ============================================================
# 26. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(

    y_test,

    y_pred_encoded,

    labels=np.arange(NUM_CLASSES)
)


cm_df = pd.DataFrame(

    cm,

    index=classes,

    columns=classes
)


cm_df.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "confusion_matrix.csv"
    )
)


print(
    "\nConfusion matrix:",
    cm.shape
)


# ============================================================
# 27. LIGHTGBM FEATURE IMPORTANCE
# ============================================================

lgb_importance = pd.DataFrame({

    "feature": X_train.columns,

    "lightgbm_importance":
        model.feature_importances_

})


lgb_importance = (

    lgb_importance

    .sort_values(
        "lightgbm_importance",
        ascending=False
    )

    .reset_index(drop=True)
)


lgb_importance.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "lightgbm_feature_importance.csv"
    ),

    index=False
)


print("\nTop LightGBM features:")

print(
    lgb_importance.head(30).to_string(
        index=False
    )
)


# ============================================================
# 28. SHAP SAMPLE
# ============================================================
#
# IMPORTANT:
#
# We intentionally use only a small SHAP sample.
#
# ~500 classes × ~1,000 features × every row can consume
# enormous memory.
#
# ============================================================

if len(X_test) > SHAP_SAMPLE_SIZE:

    X_shap = X_test.sample(

        n=SHAP_SAMPLE_SIZE,

        random_state=RANDOM_STATE
    )

else:

    X_shap = X_test.copy()


print("\n" + "=" * 90)
print("SHAP")
print("=" * 90)


print(
    "SHAP rows:",
    len(X_shap)
)

print(
    "SHAP features:",
    X_shap.shape[1]
)


# ============================================================
# 29. CREATE SHAP EXPLAINER
# ============================================================

print(
    "\nCreating SHAP TreeExplainer..."
)


explainer = shap.TreeExplainer(
    model
)


# ============================================================
# 30. SHAP VALUES
# ============================================================
#
# For a 500-class model, SHAP versions can return:
#
#   list:
#       [class][rows][features]
#
# OR:
#
#   ndarray:
#       [rows][features][classes]
#
# OR newer SHAP versions can return an Explanation object.
#
# ============================================================

print(
    "\nCalculating SHAP values..."
)


raw_shap = explainer.shap_values(
    X_shap
)


# ============================================================
# 31. CONVERT SHAP FORMAT
# ============================================================

print(
    "\nRaw SHAP type:",
    type(raw_shap)
)


if isinstance(
    raw_shap,
    list
):

    # Old SHAP format:
    #
    # list of:
    #     rows × features
    #
    shap_array = np.stack(
        raw_shap,
        axis=2
    )


elif isinstance(
    raw_shap,
    np.ndarray
):

    shap_array = raw_shap


elif hasattr(
    raw_shap,
    "values"
):

    shap_array = np.asarray(
        raw_shap.values
    )


else:

    raise ValueError(
        f"Unsupported SHAP output type: "
        f"{type(raw_shap)}"
    )


print(
    "SHAP shape:",
    shap_array.shape
)


# ============================================================
# 32. FIX POSSIBLE SHAP DIMENSION ORDER
# ============================================================

n_samples = len(X_shap)

n_features = X_shap.shape[1]

n_classes = NUM_CLASSES


if shap_array.ndim != 3:

    raise ValueError(
        f"Expected 3-dimensional SHAP output, "
        f"got {shap_array.shape}"
    )


# Expected:
#
# samples × features × classes
#

if shap_array.shape == (
    n_samples,
    n_features,
    n_classes
):

    pass


# Some versions can produce:
#
# samples × classes × features
#

elif shap_array.shape == (
    n_samples,
    n_classes,
    n_features
):

    shap_array = np.transpose(
        shap_array,
        (0, 2, 1)
    )


# Some old versions:
#
# classes × samples × features
#

elif shap_array.shape == (
    n_classes,
    n_samples,
    n_features
):

    shap_array = np.transpose(
        shap_array,
        (1, 2, 0)
    )


else:

    raise ValueError(

        f"""
Unexpected SHAP shape:

{shap_array.shape}

Expected one of:

({n_samples}, {n_features}, {n_classes})

({n_samples}, {n_classes}, {n_features})

({n_classes}, {n_samples}, {n_features})
"""

    )


print(
    "Final SHAP shape:",
    shap_array.shape
)


# ============================================================
# 33. GLOBAL SHAP IMPORTANCE
# ============================================================
#
# Average absolute SHAP across:
#
#   rows
#   product classes
#
# ============================================================

abs_shap = np.abs(
    shap_array
)


global_mean_abs = abs_shap.mean(
    axis=(0, 2)
)


global_max_abs = abs_shap.max(
    axis=(0, 2)
)


global_importance = pd.DataFrame({

    "feature":
        X_shap.columns,

    "mean_abs_shap":
        global_mean_abs,

    "max_abs_shap":
        global_max_abs

})


global_importance = (

    global_importance

    .sort_values(
        "mean_abs_shap",
        ascending=False
    )

    .reset_index(drop=True)
)


global_importance["rank"] = (
    np.arange(
        1,
        len(global_importance) + 1
    )
)


global_importance.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "global_shap_importance.csv"
    ),

    index=False
)


print("\n" + "=" * 90)
print("GLOBAL SHAP IMPORTANCE")
print("=" * 90)


print(

    global_importance.head(
        TOP_GLOBAL_FEATURES
    ).to_string(
        index=False
    )

)


# ============================================================
# 34. GLOBAL SHAP BAR
# ============================================================

top_global = (

    global_importance

    .head(
        TOP_GLOBAL_FEATURES
    )

    .sort_values(
        "mean_abs_shap"
    )
)


plt.figure(
    figsize=(12, 10)
)


plt.barh(

    top_global["feature"],

    top_global["mean_abs_shap"]

)


plt.xlabel(
    "Mean |SHAP value|"
)


plt.ylabel(
    "Feature"
)


plt.title(
    "Global SHAP Importance - All Products"
)


plt.tight_layout()


plt.savefig(

    os.path.join(
        OUTPUT_DIR,
        "global_shap_importance.png"
    ),

    dpi=300,

    bbox_inches="tight"
)


plt.show()


# ============================================================
# 35. COMPARE LIGHTGBM VS SHAP
# ============================================================

importance_comparison = (

    global_importance

    .merge(

        lgb_importance,

        on="feature",

        how="left"

    )

)


importance_comparison.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "lightgbm_vs_shap.csv"
    ),

    index=False
)


# ============================================================
# 36. SELECT TOP PRODUCTS
# ============================================================
#
# We use the most frequent products in the SHAP sample.
#
# ============================================================

shap_original_y = y.loc[
    X_shap.index
]


products_to_analyze = (

    shap_original_y

    .value_counts()

    .head(
        TOP_PRODUCTS_TO_ANALYZE
    )

    .index

)


print("\n" + "=" * 90)
print("PRODUCTS FOR DETAILED SHAP")
print("=" * 90)


print(
    products_to_analyze.tolist()
)


# ============================================================
# 37. CLASS-SPECIFIC SHAP IMPORTANCE
# ============================================================

records = []


for class_index, product in enumerate(classes):

    class_shap = shap_array[
        :,
        :,
        class_index
    ]


    mean_abs = np.abs(
        class_shap
    ).mean(
        axis=0
    )


    mean_signed = class_shap.mean(
        axis=0
    )


    temp = pd.DataFrame({

        "product_id":
            product,

        "feature":
            X_shap.columns,

        "mean_abs_shap":
            mean_abs,

        "mean_shap":
            mean_signed

    })


    temp = temp.sort_values(

        "mean_abs_shap",

        ascending=False

    )


    records.append(
        temp
    )


class_importance = pd.concat(
    records,
    ignore_index=True
)


class_importance.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "product_class_shap_importance.csv"
    ),

    index=False
)


# ============================================================
# 38. TOP FEATURES FOR SELECTED PRODUCTS
# ============================================================

top_product_features = (

    class_importance

    .sort_values(

        [
            "product_id",
            "mean_abs_shap"
        ],

        ascending=[
            True,
            False
        ]

    )

    .groupby(
        "product_id"
    )

    .head(
        TOP_FEATURES_PER_PRODUCT
    )

)


top_product_features.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "top_features_per_product.csv"
    ),

    index=False
)


# ============================================================
# 39. PRINT TOP FEATURES FOR PRODUCTS
# ============================================================

print("\n" + "=" * 90)
print("TOP FEATURES PER PRODUCT")
print("=" * 90)


for product in products_to_analyze:

    temp = (

        class_importance[
            class_importance["product_id"] == product
        ]

        .head(
            TOP_FEATURES_PER_PRODUCT
        )

    )


    print(
        f"\nProduct ID: {product}"
    )


    print(

        temp[
            [
                "feature",
                "mean_abs_shap",
                "mean_shap"
            ]

        ].to_string(
            index=False
        )

    )


# ============================================================
# 40. PRODUCT-SPECIFIC SHAP PLOTS
# ============================================================

for product in products_to_analyze:

    matching = np.where(
        classes == product
    )[0]


    if len(matching) == 0:
        continue


    class_index = matching[0]


    class_shap = shap_array[
        :,
        :,
        class_index
    ]


    plt.figure(
        figsize=(12, 10)
    )


    shap.summary_plot(

        class_shap,

        X_shap,

        max_display=TOP_FEATURES_PER_PRODUCT,

        show=False

    )


    plt.title(
        f"SHAP Summary - Product {product}"
    )


    plt.tight_layout()


    safe_product = str(product).replace(
        "/",
        "_"
    )


    plt.savefig(

        os.path.join(

            OUTPUT_DIR,

            f"shap_product_{safe_product}.png"

        ),

        dpi=300,

        bbox_inches="tight"

    )


    plt.show()


# ============================================================
# 41. PRODUCT FEATURE IMPACT
# ============================================================

impact_records = []


for class_index, product in enumerate(classes):

    class_shap = shap_array[
        :,
        :,
        class_index
    ]


    mean_shap = class_shap.mean(
        axis=0
    )


    mean_abs_shap = np.abs(
        class_shap
    ).mean(
        axis=0
    )


    temp = pd.DataFrame({

        "product_id":
            product,

        "feature":
            X_shap.columns,

        "mean_shap":
            mean_shap,

        "mean_abs_shap":
            mean_abs_shap

    })


    temp["impact"] = np.where(

        temp["mean_shap"] > 0,

        "Pushes toward product",

        "Pushes away from product"

    )


    impact_records.append(
        temp
    )


product_impact = pd.concat(

    impact_records,

    ignore_index=True
)


product_impact.to_csv(

    os.path.join(

        OUTPUT_DIR,

        "product_feature_shap_impact.csv"

    ),

    index=False

)


# ============================================================
# 42. LOCAL EXPLANATION
# ============================================================
#
# Explain one prediction.
#
# ============================================================

local_position = 0


local_index = X_shap.index[
    local_position
]


X_single = X_shap.iloc[
    [local_position]
]


actual_product = y.loc[
    local_index
]


predicted_encoded = model.predict(
    X_single
)[0]


predicted_product = label_encoder.inverse_transform(
    [int(predicted_encoded)]
)[0]


prediction_probabilities = model.predict_proba(
    X_single
)[0]


predicted_class_index = int(
    np.argmax(
        prediction_probabilities
    )
)


predicted_probability = (
    prediction_probabilities[
        predicted_class_index
    ]
)


print("\n" + "=" * 90)
print("LOCAL EXPLANATION")
print("=" * 90)


print(
    "Actual product:",
    actual_product
)


print(
    "Predicted product:",
    predicted_product
)


print(
    "Prediction confidence:",
    round(
        predicted_probability,
        6
    )
)


# ============================================================
# 43. LOCAL SHAP VALUES
# ============================================================

single_class_shap = shap_array[

    local_position,

    :,

    predicted_class_index

]


# ============================================================
# 44. LOCAL SHAP TABLE
# ============================================================

local_explanation = pd.DataFrame({

    "feature":
        X_shap.columns,

    "feature_value":
        X_single.iloc[0].values,

    "shap_value":
        single_class_shap

})


local_explanation["abs_shap"] = np.abs(
    local_explanation["shap_value"]
)


local_explanation = (

    local_explanation

    .sort_values(
        "abs_shap",
        ascending=False
    )

)


local_explanation.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "local_shap_explanation.csv"
    ),

    index=False
)


print(
    "\nTop local features:"
)


print(

    local_explanation.head(
        20
    ).to_string(
        index=False
    )

)


# ============================================================
# 45. LOCAL WATERFALL
# ============================================================

expected_value = explainer.expected_value


if isinstance(
    expected_value,
    (list, np.ndarray)
):

    base_value = expected_value[
        predicted_class_index
    ]

else:

    expected_value = np.asarray(
        expected_value
    )

    if expected_value.ndim == 0:

        base_value = expected_value.item()

    else:

        base_value = expected_value[
            predicted_class_index
        ]


single_explanation = shap.Explanation(

    values=single_class_shap,

    base_values=base_value,

    data=X_single.iloc[0].values,

    feature_names=X_shap.columns

)


plt.figure(
    figsize=(12, 10)
)


shap.plots.waterfall(

    single_explanation,

    max_display=20,

    show=False

)


plt.tight_layout()


plt.savefig(

    os.path.join(
        OUTPUT_DIR,
        "local_waterfall.png"
    ),

    dpi=300,

    bbox_inches="tight"

)


plt.show()


# ============================================================
# 46. SAVE SHAP VALUES FOR TOP PRODUCTS
# ============================================================

for product in products_to_analyze:

    matching = np.where(
        classes == product
    )[0]


    if len(matching) == 0:
        continue


    class_index = matching[0]


    product_shap_df = pd.DataFrame(

        shap_array[
            :,
            :,
            class_index
        ],

        columns=X_shap.columns,

        index=X_shap.index

    )


    safe_product = str(product).replace(
        "/",
        "_"
    )


    product_shap_df.to_csv(

        os.path.join(

            OUTPUT_DIR,

            f"shap_values_product_{safe_product}.csv"

        ),

        index=True

    )


# ============================================================
# 47. FINAL FEATURE RANKING
# ============================================================

global_importance[
    [
        "rank",
        "feature",
        "mean_abs_shap",
        "max_abs_shap"
    ]
].to_csv(

    os.path.join(

        OUTPUT_DIR,

        "final_global_feature_ranking.csv"

    ),

    index=False

)


# ============================================================
# 48. SAVE MODEL
# ============================================================

model.booster_.save_model(

    os.path.join(
        OUTPUT_DIR,
        "lightgbm_product_model.txt"
    )

)


# ============================================================
# 49. FINAL SUMMARY
# ============================================================

print("\n\n")


print("=" * 90)
print("FINAL SUMMARY")
print("=" * 90)


print(
    "\nSource dataframe:",
    selected_name
)


print(
    "Original rows:",
    len(selected_df)
)


print(
    "Rows used:",
    len(df)
)


print(
    "Product classes:",
    NUM_CLASSES
)


print(
    "Features:",
    X.shape[1]
)


print(
    "Training rows:",
    len(X_train)
)


print(
    "Testing rows:",
    len(X_test)
)


print(
    "SHAP rows:",
    len(X_shap)
)


print(
    "\nAccuracy:",
    round(
        accuracy,
        4
    )
)


print(
    "Balanced Accuracy:",
    round(
        balanced_accuracy,
        4
    )
)


print(
    "\nTop global SHAP features:"
)


print(

    global_importance.head(20)[

        [
            "rank",
            "feature",
            "mean_abs_shap"

        ]

    ].to_string(
        index=False
    )

)


print("\n")


print("=" * 90)
print("FILES SAVED")
print("=" * 90)


print(
    f"\n{OUTPUT_DIR}/"
)


print(
    """
1. classification_report.txt
2. confusion_matrix.csv
3. test_predictions.csv
4. lightgbm_feature_importance.csv
5. global_shap_importance.csv
6. final_global_feature_ranking.csv
7. lightgbm_vs_shap.csv
8. product_class_shap_importance.csv
9. product_feature_shap_impact.csv
10. top_features_per_product.csv
11. local_shap_explanation.csv
12. global_shap_importance.png
13. local_waterfall.png
14. shap_product_<product>.png
15. shap_values_product_<product>.csv
16. lightgbm_product_model.txt
"""
)


print("\nSHAP ANALYSIS COMPLETED SUCCESSFULLY.")

MULTICLASS LIGHTGBM + SHAP
PRODUCT PREDICTION

SEARCHING FOR VALID DATAFRAME

sales: shape = (0, 1030)
   product_id non-null = 0
   product_id unique = 0

cpd: shape = (100, 1032)
   product_id non-null = 100
   product_id unique = 63

>>> SELECTED DATAFRAME: cpd

Using dataframe: cpd
Shape: (100, 1032)

DATA VALIDATION

Original shape: (100, 1032)
Product ID dtype: int64
Product ID non-null: 100
Product ID null: 0
Number of products: 63

After removing NULL product_id:
Rows: 100
Products: 63

PRODUCT CLASS DISTRIBUTION

Number of product classes: 63

Smallest classes:
product_id
7      1
187    1
73     1
8      1
416    1
209    1
424    1
409    1
308    1
161    1
153    1
408    1
159    1
418    1
60     1
191    1
279    1
105    1
319    1
57     1
Name: count, dtype: int64

Largest classes:
product_id
137    2
266    2
422    2
432    2
148    2
246    2
192    2
425    2
396    2
110    2
267    2
235    2
377    2
358    2
134    2
36     3
226    3
297    3
47     3
94    

ValueError: The test_size = 14 should be greater or equal to the number of classes = 30

In [ ]:
#%pip install matplotlib shap lightgbm scikit-learn